In [ ]:
# -*- coding: utf-8 -*-

import pathlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

HERE = pathlib.Path(__file__).resolve().parent
DATA = pd.read_csv(HERE / 'dc_gain_by_series.csv')
OUT = HERE / 'output'

SERIES = {
    'Gunwi_Fuji': 'Gunwi · Fuji', 'Gunwi_Hongro': 'Gunwi · Hongro',
    'Naju_Shingo': 'Naju · Shingo', 'Naju_Wonhwang': 'Naju · Wonhwang',
    'Chuncheon_CE': 'Chuncheon · Campbell Early', 'Chuncheon_K': 'Chuncheon · Kyoho',
    'Naju_CE': 'Naju · Campbell Early', 'Naju_K': 'Naju · Kyoho',
    'Okcheon_CE': 'Okcheon · Campbell Early', 'Okcheon_K': 'Okcheon · Kyoho',
    'Daegu_Cherry': 'Daegu · Yoshino cherry', 'Gangneung_Cherry': 'Gangneung · Yoshino cherry',
    'Jeonju_Cherry': 'Jeonju · Yoshino cherry',
}

# muted single-hue ramp: short time step light, long time step dark
CMAP = LinearSegmentedColormap.from_list(
    'dusk', ['#EED3A6', '#DDA184', '#B67A8C', '#7D6A96', '#3F5878'])
INK, GREY, SOFT = '#333A40', '#777F8D', '#D5D8DC'
R_TICKS, R_LABELS = [1, 168, 336, 504, 672], ['1 h', '1 wk', '2 wk', '3 wk', '4 wk']
XLAB, YLAB = 'Calibration (d)', 'Validation (d)'
LEGEND = [Line2D([], [], color=GREY, lw=1, ls=(0, (3, 2)), label='Equal effect')]

plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': INK, 'xtick.color': INK, 'ytick.color': INK})

# one common axis range for every panel, so the 13 panels can be compared
_v = DATA[['calibration_d', 'validation_d']].to_numpy()
LIM = (np.floor(np.nanmin(_v) / 5) * 5 - 2, np.ceil(np.nanmax(_v) / 5) * 5 + 2)


def draw(ax, series, small=False):
    """Draw one series: one point per time step r."""
    d = DATA[DATA.series == series]
    lo, hi = LIM
    ax.plot([lo, hi], [lo, hi], color=GREY, lw=.8, ls=(0, (3, 2)), zorder=1)   # 1:1 line
    ax.axhline(0, color=SOFT, lw=.6, zorder=0)
    ax.axvline(0, color=SOFT, lw=.6, zorder=0)
    sc = ax.scatter(d.calibration_d, d.validation_d, c=d.r_hours, cmap=CMAP, vmin=1, vmax=672,
                    s=3 if small else 7, lw=0, alpha=.9, zorder=3)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal', adjustable='box')
    ticks = np.arange(-20, hi, 10)
    ax.set_xticks(ticks); ax.set_yticks(ticks)
    ax.set_title(SERIES[series], loc='left', fontsize=10 if small else 12, color=INK, pad=3)
    return sc


def colorbar(fig, sc, rect):
    cb = fig.colorbar(sc, cax=fig.add_axes(rect), orientation='horizontal')
    cb.set_ticks(R_TICKS); cb.set_ticklabels(R_LABELS); cb.outline.set_visible(False)
    cb.set_label('Time step', fontsize=9)


def singles():
    """One figure per series (13 figures) from the same function."""
    d = OUT / 'by_series'; d.mkdir(parents=True, exist_ok=True)
    for s in SERIES:
        fig = plt.figure(figsize=(4.4, 5.2))
        ax = fig.add_axes([.17, .27, .78, .62])
        sc = draw(ax, s)
        ax.set_xlabel(XLAB); ax.set_ylabel(YLAB)
        ax.legend(handles=LEGEND, loc='upper left', frameon=False, fontsize=8.5)
        colorbar(fig, sc, [.25, .08, .6, .025])
        fig.savefig(d / f'dc_scatter_{s}.png', dpi=250); plt.close(fig)


def grid():
    """All 13 series in one figure (3 x 5), same function."""
    fig, axs = plt.subplots(3, 5, figsize=(13, 8.8), sharex=True, sharey=True)
    fig.subplots_adjust(left=.07, right=.99, top=.97, bottom=.08, wspace=.12, hspace=.28)
    sc = None
    for k, ax in enumerate(axs.flat):
        if k >= len(SERIES):
            ax.axis('off'); continue
        sc = draw(ax, list(SERIES)[k], small=True)
        ax.tick_params(labelbottom=True, labelleft=True, labelsize=8)
    axs.flat[13].legend(handles=LEGEND, loc='upper left', bbox_to_anchor=(0, .85),
                        frameon=False, fontsize=11)
    colorbar(fig, sc, [.64, .17, .3, .02])
    fig.supxlabel(XLAB, fontsize=11, color=INK)
    fig.supylabel(YLAB, fontsize=11, color=INK)
    fig.savefig(OUT / 'grid_dc_scatter.png', dpi=220); plt.close(fig)


if __name__ == '__main__':
    OUT.mkdir(parents=True, exist_ok=True)
    singles()
    grid()
    print('saved to', OUT)